# Hausa Crisis Signal Detector Training
Fine-tuning AfriBERTa for humanitarian crisis classification in Hausa.

**Author:** Sadiya Muhammad Kilgori
**Repo:** [hausa-crisis-signal-detector](https://github.com/SKilgori/hausa-crisis-signal-detector)

In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas torch -q

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# Load data from GitHub
DATA_URL = "https://raw.githubusercontent.com/SKilgori/hausa-crisis-signal-detector/main/data/hausa_crisis_data.csv"
df = pd.read_csv(DATA_URL)

LABELS = ["conflict", "displacement", "disease_outbreak", "flood", "food_insecurity", "no_crisis"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for idx, label in enumerate(LABELS)}

df["label_id"] = df["label"].map(label2id)
print(f"Total samples: {len(df)}")
print(df["label"].value_counts())

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_dataset = Dataset.from_pandas(train_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))
test_dataset = Dataset.from_pandas(test_df[["text", "label_id"]].rename(columns={"label_id": "labels"}))

MODEL_NAME = "castorini/afriberta_large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding=True, truncation=True, max_length=128)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./hausa_crisis_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=10,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
from huggingface_hub import notebook_login
print("Please log in to Hugging Face to push the model.")
notebook_login()

REPO_ID = "Skilgori/hausa-crisis-signal-detector"
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)